# Research Results and Statistical Analysis

## Comparative Evaluation of Machine Learning Models
## for Flow-Based DDoS Attack Detection

This notebook presents the reproducible experimental
analysis of machine learning benchmark measurements.

The analysis includes:

- Raw data ingestion.
- Warm-up filtering.
- Data quality validation.
- Error-rate monitoring.
- Statistical aggregation.
- Median, p95 and p99 calculation.
- Bootstrap 95% confidence intervals.
- Scientific visualization.
- Exporting processed results.

Random seed: 42.

In [ ]:
import sys
import json
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

current = Path.cwd().resolve()

ROOT = None

for candidate in [current, *current.parents]:
    if (
        (candidate / "data").exists()
        and (candidate / "src").exists()
    ):
        ROOT = candidate
        break

if ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root."
    )

RAW_PATH = ROOT / "data/raw/test_run.jsonl"

PROCESSED_PATH = (
    ROOT / "data/processed/metrics_summary.csv"
)

FIGURES_PATH = ROOT / "reports/figures"

FIGURES_PATH.mkdir(
    parents=True,
    exist_ok=True
)

sns.set_theme(
    style="whitegrid",
    context="paper",
    palette="colorblind"
)

print("Project root:", ROOT)
print("Random seed:", SEED)
print("Raw data:", RAW_PATH)

## 1. Data Ingestion

The raw benchmark data is stored in JSON Lines format.

Each line represents one benchmark measurement.

The dataset contains warm-up and evaluation records.

Warm-up records are excluded from the final analysis
to reduce the influence of initial execution overhead.

In [ ]:
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Raw data not found: {RAW_PATH}"
    )

records = []

with RAW_PATH.open("r", encoding="utf-8") as file:

    for line_number, line in enumerate(file, start=1):

        if not line.strip():
            continue

        try:
            records.append(json.loads(line))

        except json.JSONDecodeError as error:

            raise ValueError(
                f"Invalid JSON at line {line_number}"
            ) from error

df = pd.DataFrame(records)

print("Total raw records:", len(df))

print("\nAvailable columns:")
print(df.columns.tolist())

display(df.head())

## 2. Warm-up Filtering

The initial execution measurements are classified
as warm-up runs.

These records are excluded from statistical analysis.

Only records with phase = evaluation and
is_warmup = False are retained.

In [ ]:
def to_bool(series):

    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )


df["is_warmup"] = to_bool(df["is_warmup"])

df["error"] = to_bool(df["error"])

df["phase"] = (
    df["phase"]
    .astype(str)
    .str.lower()
)

warmup_count = (
    (df["phase"] == "warmup") |
    df["is_warmup"]
).sum()

df_eval = df[
    (df["phase"] == "evaluation") &
    (~df["is_warmup"])
].copy()

print("Total raw records:", len(df))

print("Warm-up records removed:", int(warmup_count))

print("Evaluation records:", len(df_eval))

assert len(df_eval) > 0

assert not df_eval["is_warmup"].any()

display(df_eval.head())

## 3. Data Quality and Error Analysis

Failed inference measurements are monitored
as a guardrail metric.

The error rate is calculated as:

Error Rate = Failed Runs / Total Evaluation Runs

Only successful inference measurements are used
for latency and throughput calculations.

In [ ]:
total_runs = len(df_eval)

failed_runs = int(
    df_eval["error"].sum()
)

error_rate = (
    failed_runs / total_runs
    if total_runs > 0
    else np.nan
)

print("Total evaluation runs:", total_runs)

print("Failed evaluation runs:", failed_runs)

print(f"Overall error rate: {error_rate:.4%}")

error_summary = (
    df_eval.groupby("model")
    .agg(
        total_runs=("error", "count"),
        errors=("error", "sum")
    )
)

error_summary["error_rate"] = (
    error_summary["errors"] /
    error_summary["total_runs"]
)

display(error_summary)

## 4. Statistical Aggregation

The following statistics are calculated:

- Median latency.
- 95th percentile latency.
- 99th percentile latency.
- Bootstrap 95% confidence interval.
- Median throughput.
- Classification metrics.
- Error rate.

The bootstrap procedure uses 5000 resamples
and a fixed random seed of 42.

In [ ]:
def bootstrap_ci(
    values,
    iterations=5000,
    statistic=np.median
):

    values = np.asarray(
        values,
        dtype=float
    )

    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan

    if len(values) == 1:
        return float(values[0]), float(values[0])

    rng = np.random.default_rng(SEED)

    samples = rng.choice(
        values,
        size=(iterations, len(values)),
        replace=True
    )

    estimates = statistic(
        samples,
        axis=1
    )

    return (
        float(np.percentile(estimates, 2.5)),
        float(np.percentile(estimates, 97.5))
    )


successful = df_eval[
    ~df_eval["error"]
].copy()

successful["latency_ms"] = pd.to_numeric(
    successful["latency_ms"],
    errors="coerce"
)

successful["throughput_rps"] = pd.to_numeric(
    successful["throughput_rps"],
    errors="coerce"
)

successful = successful.replace(
    [np.inf, -np.inf],
    np.nan
)

successful = successful.dropna(
    subset=["latency_ms", "throughput_rps"]
)

latency_results = []

for (model, batch), group in successful.groupby(
    ["model", "batch_size"]
):

    values = group["latency_ms"].to_numpy()

    low, high = bootstrap_ci(values)

    latency_results.append({

        "model": model,

        "batch_size": batch,

        "count": len(values),

        "median_latency_ms": np.median(values),

        "p95_latency_ms": np.percentile(values, 95),

        "p99_latency_ms": np.percentile(values, 99),

        "ci95_low": low,

        "ci95_high": high

    })

latency_summary = pd.DataFrame(
    latency_results
)

display(latency_summary)

In [ ]:
classification_results = []

for model, group in df_eval.groupby("model"):

    folds = group.drop_duplicates(
        subset=["fold"]
    )

    for metric in [
        "accuracy",
        "precision",
        "recall",
        "f1_score",
        "fpr"
    ]:

        values = (
            pd.to_numeric(
                folds[metric],
                errors="coerce"
            )
            .dropna()
            .to_numpy()
        )

        if len(values) == 0:
            continue

        low, high = bootstrap_ci(
            values,
            statistic=np.mean
        )

        classification_results.append({

            "model": model,

            "metric": metric,

            "mean": np.mean(values),

            "ci95_low": low,

            "ci95_high": high

        })

classification_summary = pd.DataFrame(
    classification_results
)

display(classification_summary)

## 5. Visualization

The first figure presents median inference latency
under different batch multipliers.

The shaded region represents the bootstrap
95% confidence interval of the median.

The second figure compares latency distributions
at a fixed batch multiplier of 1.

Both figures are exported in PDF and PNG formats.

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(ROOT / "src/process_data.py")
    ],
    check=True,
    cwd=ROOT
)

final_summary = pd.read_csv(
    PROCESSED_PATH
)

print("Processed statistical summary:")

display(final_summary)

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(ROOT / "src/visualize.py")
    ],
    check=True,
    cwd=ROOT
)

print("Generated figures:")

for figure in sorted(FIGURES_PATH.iterdir()):

    if figure.suffix in [".pdf", ".png"]:

        print(
            figure.name,
            f"({figure.stat().st_size:,} bytes)"
        )

## 6. Results and Discussion

The experiment compares four machine learning
classifiers in terms of computational performance
and classification quality.

The statistical analysis includes median latency,
p95, p99, throughput and error rates.

Warm-up measurements are excluded from the analysis.

The results should be interpreted with caution.

The original classification dataset contains only
ten synthetic observations. Therefore, the experiment
does not establish reliable real-world DDoS detection
performance.

Larger and more representative network traffic
datasets are required for further validation.

## 7. Reproducibility

The analysis uses:

- Python.
- NumPy.
- Pandas.
- Matplotlib.
- Seaborn.
- Bootstrap resampling.
- Random seed 42.

The raw data, processing scripts and generated figures
are stored in the project repository.